In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import mode
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.utils import shuffle

In [2]:
seed = 1234
np.random.seed(seed)

## Data Loading

In [3]:
DATA = pd.read_csv("data/proc_training_2nd-thres10-withpub.csv")
PUB_DATA = pd.read_csv("data/proc_public_2nd-thres10-withpub.csv")
TEST_DATA = pd.read_csv("data/proc_testing_2nd-thres10-withpub.csv")
PRIV_DATA = pd.read_csv("data/proc_private_2nd-thres10-withpub.csv")

In [4]:
DATA = pd.concat((DATA, PUB_DATA))
X = DATA.drop(columns='label')
y = DATA['label']

In [5]:
# X_train = DATA.drop(columns='label')
# y_train = DATA['label']
# X_valid = PUB_DATA.drop(columns='label')
# y_valid = PUB_DATA['label']

In [6]:
TEST_txkey = TEST_DATA['txkey']
X_test = TEST_DATA.drop(columns='txkey')

PRIV_txkey = PRIV_DATA['txkey']
X_priv = PRIV_DATA.drop(columns='txkey')

## Model Construction

In [7]:
# model = lgb.LGBMClassifier(
#     boosting_type='gbdt',
#     objective='binary',
#     device='gpu',
#     random_state=seed,
# )
# model.set_params(
#     bagging_freq=1,
#     bagging_fraction=0.9,
#     feature_fraction=0.9,
#     # feval=lgb_f1_score,
#     # evals_result=eval_res,
# )

## Training & Validation
先 grid search 找到不錯的參數，再去做 cross validation

In [8]:
cat_feats = [
    'contp',
    'etymd',
    'ecfg',
    # 'insfg',
    'bnsfg',
    'stscd',
    'ovrlt',
    'flbmk',
    'hcefg',
    'csmcu',
    'flg_3dsmk',
    # 'mcc_gp',
    # 'region_gp',
    # 'chid_gp'
]

# X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3)

X_pos_train, X_pos_valid, y_pos_train, y_pos_valid = train_test_split(X[y == 1], y[y == 1], test_size=0.1)
X_neg_train, X_neg_valid, y_neg_train, y_neg_valid = train_test_split(X[y == 0], y[y == 0], test_size=0.1)

X_train = pd.concat((X_pos_train, X_neg_train))
y_train = pd.concat((y_pos_train, y_neg_train))

X_valid = pd.concat((X_pos_valid, X_neg_valid))
y_valid = pd.concat((y_pos_valid, y_neg_valid))

X_train, y_train = shuffle(X_train, y_train)
X_valid, y_valid = shuffle(X_valid, y_valid)

# model.fit(
#     X_train, y_train,
#     categorical_feature=cat_feats,
#     callbacks=[lgb.early_stopping(5)],
#     eval_set=[(X_valid, y_valid)],
#     eval_metric=['binary_logloss'],
# )

In [ ]:
params = {
    'num_leaves': range(100, 200, 20),
    'learning_rate': [1e-2, 4e-2, 7e-2],
    'n_estimators': [100],
    'min_split_gain': [3e-3],
    'min_child_samples': range(20, 50, 10),
}
grid = ParameterGrid(params)
best_params = None
best_score = 100

for params in grid:
    model = lgb.LGBMClassifier(
        boosting_type='gbdt',
        objective='binary',
        device='gpu',
        random_state=seed,
        **params
    )
    
    model.fit(
        X_train, y_train,
        categorical_feature=cat_feats,
        callbacks=[lgb.early_stopping(5)],
        eval_set=[(X_valid, y_valid)],
        eval_metric=['binary_logloss'],
    )
    
    score = model.best_score_['valid_0']['binary_logloss']
    if score < best_score:
        best_score = score
        best_params = params

[LightGBM] [Info] Number of positive: 28025, number of negative: 7574434
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1443
[LightGBM] [Info] Number of data points in the train set: 7602459, number of used features: 18
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 8 dense feature groups (58.00 MB) transferred to GPU in 0.042100 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003686 -> initscore=-5.599437
[LightGBM] [Info] Start training from score -5.599437
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.00573543
[LightGBM] [Info] Number of positive: 28025, number of negative: 7574434
[LightGBM] [Info] This is the GPU traine

In [18]:
print(f'Best Score: {best_score}')
print(f'Best Params: {best_params}')

Best Score: 0.0032895473457825826
Best Params: {'learning_rate': 0.04, 'min_child_samples': 40, 'min_split_gain': 0.003, 'n_estimators': 100, 'num_leaves': 180}


In [ ]:
# Found in the above grid search
# best_params = {
#     'learning_rate': 0.1,
#     'min_child_samples': 20,
#     'min_split_gain': 0.0001,
#     'n_estimators': 50,
#     'num_leaves': 51
# }

In [19]:
D_train = lgb.Dataset(X_train, y_train)

In [20]:
lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'device': 'gpu',
}
lgb_params.update(best_params)
eval_res = lgb.cv(
    lgb_params,
    D_train,
    nfold=5,
    metrics='binary_log_loss',
    categorical_feature=cat_feats,
    return_cvbooster=True
)
eval_res

/home/kuanhe/.cache/pypoetry/virtualenvs/esun-YNnw7uhi-py3.11/lib/python3.11/site-packages/lightgbm/engine.py:685: UserWarning: Found 'n_estimators' in params. Will use it instead of 'num_boost_round' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'num_boost_round' argument")


[LightGBM] [Info] Number of positive: 22420, number of negative: 6059547
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1438
[LightGBM] [Info] Number of data points in the train set: 6081967, number of used features: 18
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 8 dense feature groups (46.40 MB) transferred to GPU in 0.034243 secs. 1 sparse feature groups
[LightGBM] [Info] Number of positive: 22420, number of negative: 6059547
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1438
[LightGBM] [Info] Number of data points in the train set: 6081967, number of used features: 18
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[Ligh

{'cvbooster': <lightgbm.engine.CVBooster at 0x7f1a36131c10>}

In [21]:
models = eval_res['cvbooster']

## Predict

In [22]:
outputs_test = models.predict(X_test)
outputs_priv = models.predict(X_priv)
preds_test = [np.where(outputs_test[i] < 0.5, 0, 1) for i in range(len(outputs_test))]
preds_priv = [np.where(outputs_priv[i] < 0.5, 0, 1) for i in range(len(outputs_priv))]
y_pred_test = mode(preds_test)[0]
y_pred_priv = mode(preds_priv)[0]

res = {
    'txkey': pd.concat((TEST_txkey, PRIV_txkey)),
    'label': np.concatenate((y_pred_test, y_pred_priv))
}

pd.DataFrame(res).to_csv('preds.csv', index=False)

In [23]:
pd.Series(y_pred_test).value_counts()

0    598574
1      1608
Name: count, dtype: int64

In [24]:
pd.Series(y_pred_priv).value_counts()

0    752157
1      1982
Name: count, dtype: int64